In [1]:
!pip install torchvision

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

In [3]:
backbone = torchvision.models.resnet50(weights=torchvision.models.resnet.ResNet50_Weights)

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
class CBR(nn.Module):
    def __init__(self, inp_ch, out_ch, filt_size, dilation=1):
        super().__init__()
        if filt_size == 1:
            padding = 0
        else:
            padding = dilation
        self.conv = nn.Sequential(
            nn.Conv2d(inp_ch, out_ch, filt_size, dilation=dilation, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
            )

    def forward(self, x):
        x = self.conv(x)
        return x

In [5]:
class ASPP_Block(nn.Module):
    def __init__(self, inp_ch, out_ch):
        super().__init__()
        '''
        We need the following:
        1) Average Pooling followed by CBR and then upsampling to match 28x28
        2) 1x1 conv block
        3) 1 conv block of 3x3 each with dilation rates of 6, 12, 18
        4) concat the output of 5 blocks
        5) apply 1x1 conv to donwsize the kernel maps to 256
        6) Upsample the resulting output by 4 --> 14*2*2 = 56 (provided initial input is 224, 224)
        '''
        self.avgpool_conv = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            CBR(inp_ch, out_ch, 1)
        )  #(this will be interpolated back to original shapeTBD in forward block)
        self.conv_1_1 = CBR(inp_ch, out_ch, 1)
        self.conv_3_6 = CBR(inp_ch, out_ch, 3, 6)
        self.conv_3_12 = CBR(inp_ch, out_ch, 3, 12)
        self.conv_3_18 = CBR(inp_ch, out_ch, 3, 18)
        self.conv_project = CBR(out_ch*5, out_ch, 1)

    def forward(self, x):
        input_size = x.shape[-2:]
        pool = self.avgpool_conv(x)
        pool_out = F.interpolate(pool, size=input_size, mode='bilinear', align_corners=False)

        conv_1_1_out = self.conv_1_1(x)
        conv_3_6_out = self.conv_3_6(x)
        conv_3_12_out = self.conv_3_12(x)
        conv_3_18_out = self.conv_3_18(x)
        aspp_concat = torch.cat([pool_out,
                                 conv_1_1_out,
                                 conv_3_6_out,
                                 conv_3_12_out,
                                 conv_3_18_out
                                 ], dim=1
                                )
        conv_project_out = self.conv_project(aspp_concat)
        return conv_project_out

In [6]:
class Encoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()

        self.bb = backbone
        self.initial = nn.Sequential(*list(backbone.children())[:4])
        self.layer1 = self.bb.layer1
        self.layer2 = self.bb.layer2
        self.layer3 = self.bb.layer3
        self.aspp = ASPP_Block(1024, 256)

    def forward(self, x):
        x = self.initial(x)
        decoder_input = self.layer1(x)
        x = self.layer2(decoder_input)
        x = self.layer3(x)
        x = self.aspp(x)
        return decoder_input, x

In [7]:
enc = Encoder(backbone)

In [8]:
x = torch.ones(2, 3, 224, 224)
inp, out = enc(x)
inp.shape, out.shape

(torch.Size([2, 256, 56, 56]), torch.Size([2, 256, 14, 14]))

In [9]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.init_layer = CBR(256, 48, 1)
        self.conv_layer = nn.Sequential(
            CBR(256+48, 256, 3),
            CBR(256, 256, 3)
        )

    def forward(self, decoder_input, encoder_output):
        init_layer_out = self.init_layer(decoder_input)
        encoder_output_exp = F.interpolate(encoder_output, size=init_layer_out.size()[2:], mode='bilinear', align_corners=False)
        x = torch.cat([encoder_output_exp, init_layer_out], dim=1)
        conv_layer_out = self.conv_layer(x)
        return conv_layer_out

In [10]:
class DeepLabV3plus(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()

        self.encoder = Encoder(backbone)
        self.decoder = Decoder()
        self.conv_out = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        input_size = x.shape[-2:]
        decoder_input, encoder_output = self.encoder(x)
        decoder_output = self.decoder(decoder_input, encoder_output)
        out = F.interpolate(decoder_output, size=input_size, mode='bilinear', align_corners=False)
        out = self.conv_out(out)
        return out

In [11]:
deepv3plus = DeepLabV3plus(backbone, 9)

In [12]:
deepv3plus

DeepLabV3plus(
  (encoder): Encoder(
    (bb): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace

In [13]:
x = torch.ones(2, 3, 224, 224)
x = deepv3plus(x)
x.shape

torch.Size([2, 9, 224, 224])

In [15]:
x

tensor([[[[0.0286, 0.0000, 0.0807,  ..., 0.1320, 0.0087, 0.0000],
          [0.0000, 0.0000, 0.0988,  ..., 0.2770, 0.0180, 0.0170],
          [0.0612, 0.0696, 0.0000,  ..., 0.0908, 0.0037, 0.3106],
          ...,
          [0.0368, 0.1006, 0.0000,  ..., 0.1182, 0.1081, 0.2348],
          [0.0000, 0.0000, 0.0000,  ..., 0.1885, 0.0000, 0.0000],
          [0.0197, 0.0000, 0.0898,  ..., 0.1612, 0.0000, 0.0000]],

         [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0992, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.2599],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.2429]],

         [[0.3324, 0.0433, 0.0172,  ..., 0.0000, 0.0000, 0.5778],
          [0.8321, 0.1226, 0.0879,  ..., 0.2276, 0.4332, 0.1507],
          [0.6637, 0.5474, 0.2257,  ..., 0

In [14]:
x = torch.ones(1, 3, 224, 224)
initial = nn.Sequential(*list(backbone.children())[:4])
x = initial(x)
p_1= backbone.layer1(x)
print(p_1.shape)
x = backbone.layer2(p_1)
x = backbone.layer3(x)
x.shape

torch.Size([1, 256, 56, 56])


torch.Size([1, 1024, 14, 14])

In [1]:
!nvidia-smi

Mon Jul 21 14:27:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:00:1E.0 Off |                    0 |
| N/A   44C    P0             33W /   70W |   13907MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----